# 문장에서 앱 단어 잘라내기 — 11차 whisper 정렬 + 재받아쓰기 (TODOS 3b)

**왜.** 12차·A-1·목표 단어 검증이 모두 "낯선 앱 단어는 그 단어의 단독 예시가 있어야 는다"고
했다. 미사용 TS01 문장 정렬(`align_ts01rest_colab.ipynb`)로 문장 세그먼트 20,574개(105시간)를
얻었다. 이 노트북은 그 안의 앱 단어 구간을 찾는다.

**어떻게** (`cut_words_whisper.py`, 대장 「문장에서 단어 잘라내기」)
1. 11차(LoRA 병합)에 정답 문장을 teacher forcing으로 넣고 교차 어텐션 DTW로 토큰 시각
2. 단어 앞 침묵을 에너지로 깎고 끝은 말소리가 이어지는 동안 늘린 뒤 앞뒤 여유
3. 잘린 구간만 11차가 다시 받아써 **그 단어가 정확히** 나와야 `keep`

**출력은 시각뿐이다** — `MyDrive/ts01rest_word_cuts.jsonl`(단어 하나 = 한 줄). 클립 오디오는
로컬에서 회수 zip으로 자른다. 드라이브에 작은 파일 수천 개를 쓰면 FUSE가 끊긴다.
줄마다 바로 써서 **세션이 끊겨도 이어서 간다**.

**준비물(드라이브 `MyDrive/`)**
- `ts01rest_segs_refined.zip` — 정렬 노트북 셀 6의 회수본(이미 있다)
- `whisper-608-lora-11차/` — 11차 어댑터(이미 있다)
- `ts01rest_cut_scripts.zip` — 스크립트 + 앱 단어 목록(새로 올린다)
- `colab_trainset_big7.zip` — 셀 4.5 관문용 단어 클립(있으면 쓴다)

## 셀 1 — GPU 확인 (T4)

In [ ]:
!nvidia-smi

## 셀 2 — 설치 (환경 핀). numpy가 바뀔 때만 재시작을 요구한다

In [ ]:
# ══ 환경 고정 ═══════════════════════════════════════════════
# 회차 간 비교를 하려면 프레임워크가 회차마다 바뀌면 안 된다. 7차(8/18)는
# transformers 4.x, 7차재현(8/20)은 5.15.1이었고 그 사이에 같은 배치 설정이
# T4에서 OOM으로 죽었다. 올릴 때는 회차를 새로 시작하고 대장에 환경을 적는다.
TFV = "5.15.1"

# numpy는 Colab 이미지가 든 버전을 그대로 고정한다. 숫자를 하드코딩하면
# 이미지가 바뀔 때마다 다시 깨진다:
#   너무 높음 → AttributeError: module 'numpy' has no attribute '_no_nep50_warning'
#   너무 낮음 → ImportError: cannot import name '_center' from numpy._core.umath
import numpy, subprocess, sys
NPV = numpy.__version__
print("Colab 기본 numpy:", NPV, "→ 이 버전으로 고정 / transformers →", TFV)

# 핵심: numpy 핀을 같은 pip 호출 안에 넣어야 resolver가 도로 올리지 못한다.
!pip install -q -U "transformers=={TFV}" datasets peft accelerate evaluate jiwer librosa "numpy=={NPV}"
!pip uninstall -y -q torchao   # peft 버전검사와 충돌, LoRA엔 불필요

# **새 프로세스로** 확인한다. 이 커널이 이미 들고 있는 모듈은 pip이 디스크를
# 바꿔도 안 바뀌므로, 커널 안에서 import해 확인하면 거짓 통과한다.
_out = subprocess.run(
    [sys.executable, "-c",
     "import numpy, transformers; print(numpy.__version__, transformers.__version__)"],
    capture_output=True, text=True)
print("디스크:", (_out.stdout or _out.stderr).strip())
_disk = _out.stdout.split()

if len(_disk) != 2 or _disk[1] != TFV:
    raise SystemExit(f"transformers가 {TFV}가 아니다. 위 pip 출력을 확인해라.")
# numpy만이 재시작을 강제한다 — 이 커널이 이미 로드했기 때문이다.
# transformers·datasets 등은 아직 import 전이라 새 파일이 그대로 읽힌다.
if _disk[0] != NPV:
    raise SystemExit(f"numpy가 {NPV} → {_disk[0]}로 바뀌었다."
                     " [런타임 → 세션 다시 시작] 후 셀 2.5부터 다시 실행해라.")
print("✅ 설치 완료 — 재시작 없이 셀 2.5로 계속 진행")


## 셀 2.5 — 재시작 후 검증 (여기서 통과해야 아래가 의미 있음)
버전 3종을 찍고 실제 임포트까지 해본다. 실패하면 아래 셀들은 전부 같은 이유로 터지니 여기서 잡는다.
- `numpy has no attribute '_no_nep50_warning'` → 런타임 torch가 지금 numpy(≥2.3)를 못 따라감
- `cannot import name '_center' from numpy._core.umath` → 반대로 numpy가 너무 낮음

**처방은 둘 다 동일**: 런타임 → **세션 삭제 후 다시 시작** → 셀 2부터 다시(셀 2가 이미지 기준 numpy를 유지한다).
버전 숫자를 손으로 고쳐 맞추려 하지 말 것.

In [ ]:
import numpy, torch, transformers
print("numpy       :", numpy.__version__)
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
if not transformers.__version__.startswith("5.15"):
    raise SystemExit("transformers 핀이 안 먹었다 — 셀 2를 다시 실행해라.")

# torch↔numpy 불일치면 바로 아래 줄에서 터진다(셀 4의 그 에러와 같은 원인)
from transformers import WhisperForConditionalGeneration
import librosa; librosa.stft(numpy.zeros(2048, dtype=numpy.float32))   # _center 계열 조기 탐지
print("✅ 임포트 정상 — 셀 3으로")

## 셀 3 — 드라이브 마운트 · 세그먼트 풀기 · 대상 계획

새 세션이면 zip을 다시 푼다(수 분). 이미 풀려 있으면 건너뛴다.

In [ ]:
import os, sys, json, zipfile, time
from google.colab import drive
drive.mount("/content/drive")

D    = "/content/drive/MyDrive"
REFZ = f"{D}/ts01rest_segs_refined.zip"
SZIP = f"{D}/ts01rest_cut_scripts.zip"
ADAPTER = f"{D}/whisper-608-lora-11차"
OUT  = f"{D}/ts01rest_word_cuts.jsonl"     # 결과(이어쓰기)
REF  = "/content/ts01rest_ref"             # 세그먼트 wav (세션 로컬)
SCR  = "/content/cutw"

for p in (REFZ, SZIP, ADAPTER):
    assert os.path.exists(p), f"없다: {p}"
os.makedirs(SCR, exist_ok=True)
zipfile.ZipFile(SZIP).extractall(SCR)
sys.path.insert(0, SCR)
print(open(f"{SCR}/VERSION.txt", encoding="utf-8").read().strip())

if not os.path.exists(f"{REF}/segments.jsonl"):
    t0 = time.time()
    with zipfile.ZipFile(REFZ) as z:          # 파일별로 꺼낸다(한 번에 크게 읽다 FUSE가 끊긴 적이 있다)
        names = z.namelist()
        for k, m in enumerate(names, 1):
            z.extract(m, REF)
            if k % 5000 == 0:
                print(f"  풀기 {k}/{len(names)}", flush=True)
    print(f"풀기 {(time.time() - t0) / 60:.1f}분")

import cut_words_whisper as W
WORDS = json.load(open(f"{SCR}/app_words.json", encoding="utf-8"))
rows = [json.loads(l) for l in open(f"{REF}/segments.jsonl", encoding="utf-8") if l.strip()]
plan, skipped = W.plan_rows(rows, WORDS)
n_targets = sum(len(W.find_targets(r["reference_text"], WORDS)) for r in plan)
print(f"세그먼트 {len(rows)}개 · 앱 단어 {len(WORDS)}종")
print(f"정렬 대상 {len(plan)}개(단어 자리 {n_targets}개) · 건너뜀 {skipped}")
missing = [r for r in plan[:200] if not os.path.exists(os.path.join(REF, r["segment_wav_relpath"]))]
assert not missing, f"wav가 없다: {missing[0]['segment_wav_relpath']}"

## 셀 4 — 11차 로드 (원래 turbo + 11차 LoRA 병합)

In [ ]:
import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from peft import PeftModel

MODEL_ID = "openai/whisper-large-v3-turbo"
processor = WhisperProcessor.from_pretrained(MODEL_ID, language="ko", task="transcribe")
# fp32(평가 노트북과 같다). attn_implementation="eager" — 교차 어텐션 확률을 돌려받아야 한다
# (sdpa는 어텐션을 내지 않는다). 인코더 자기 어텐션은 받지 않으므로 메모리는 문제없다.
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.float32, attn_implementation="eager")
model = PeftModel.from_pretrained(model, ADAPTER).merge_and_unload().cuda().eval()
model.generation_config.language = "ko"
model.generation_config.task = "transcribe"
model.generation_config.max_new_tokens = 16     # 단어 하나 받아쓰기
model.generation_config.use_cache = True
assert model.generation_config.alignment_heads, "정렬 헤드가 없다"
aligner = W.TokenAligner(model, processor)
print("11차 병합 완료 · 정렬 헤드", len(aligner.heads), "개 ·", model.device)

import random, soundfile as sf
GATE_MIN = 0.85
BIG7 = f"{D}/colab_trainset_big7.zip"
if not os.path.exists(BIG7):
    print("⚠️ colab_trainset_big7.zip이 없어 관문을 건너뛴다 — 결과의 정렬 품질을 따로 확인해야 한다")
else:
    G = "/content/big7"
    with zipfile.ZipFile(BIG7) as z:
        tr = [json.loads(l) for l in z.read("train.jsonl").decode("utf-8").splitlines() if l.strip()]
        cand = [r for r in tr if r["task_type"] == "wordlist" and len(r["text"].strip()) >= 2]
        random.Random(0).shuffle(cand)        # 화자가 몰리지 않게 섞는다(앞 400개만 쓰면 87%로 떨어졌다)
        clips = []
        for r in cand:
            if len(clips) >= 400:
                break
            z.extract(r["audio"], G)
            a, sr = sf.read(f"{G}/{r['audio']}", dtype="float32")
            if sr == W.SR and 0.4 <= len(a) / sr <= 1.2:
                clips.append((a, r["text"].strip()))
    # 단어 300개 한 번. 시드별 60개는 원래 turbo에서 90·78·93%로 흔들렸다 — 문턱 판정에 못 쓴다.
    r = W.synthetic_check(aligner, clips, n_sent=60, seed=1)
    print(f"  덮음 {r['covered']:.1%} · 군더더기 중앙 {r['extra_median']:.2f}초 (n={r['n']})")
    if r["covered"] < GATE_MIN:
        raise SystemExit(f"덮음 {r['covered']:.1%} < {GATE_MIN:.0%} — 멈춘다. 결과를 대장에 적고 원인을 본다.")
    print("→ 관문 통과")

In [ ]:
import random, soundfile as sf
GATE_MIN = 0.85
BIG7 = f"{D}/colab_trainset_big7.zip"
if not os.path.exists(BIG7):
    print("⚠️ colab_trainset_big7.zip이 없어 관문을 건너뛴다 — 결과의 정렬 품질을 따로 확인해야 한다")
else:
    G = "/content/big7"
    with zipfile.ZipFile(BIG7) as z:
        tr = [json.loads(l) for l in z.read("train.jsonl").decode("utf-8").splitlines() if l.strip()]
        cand = [r for r in tr if r["task_type"] == "wordlist" and len(r["text"].strip()) >= 2]
        random.Random(0).shuffle(cand)        # 화자가 몰리지 않게 섞는다(앞 400개만 쓰면 87%로 떨어졌다)
        clips = []
        for r in cand:
            if len(clips) >= 400:
                break
            z.extract(r["audio"], G)
            a, sr = sf.read(f"{G}/{r['audio']}", dtype="float32")
            if sr == W.SR and 0.4 <= len(a) / sr <= 1.2:
                clips.append((a, r["text"].strip()))
    # 단어 300개 한 번. 시드별 60개는 원래 turbo에서 90·78·93%로 흔들렸다 — 문턱 판정에 못 쓴다.
    r = W.synthetic_check(aligner, clips, n_sent=60, seed=1)
    print(f"  덮음 {r['covered']:.1%} · 군더더기 중앙 {r['extra_median']:.2f}초 (n={r['n']})")
    if r["covered"] < GATE_MIN:
        raise SystemExit(f"덮음 {r['covered']:.1%} < {GATE_MIN:.0%} — 멈춘다. 결과를 대장에 적고 원인을 본다.")
    print("→ 관문 통과")

## 셀 5 — 잘라내기 (끊기면 셀 1·2.5·3·4를 다시 돌리고 이 셀만 다시)

세그먼트 하나를 끝낼 때마다 결과를 드라이브에 쓴다. 다시 돌리면 끝난 세그먼트는 건너뛴다.

In [ ]:
import soundfile as sf

done = set()
if os.path.exists(OUT):
    for l in open(OUT, encoding="utf-8"):
        if l.strip():
            done.add(json.loads(l)["segment"])
todo = [r for r in plan if r["segment_wav_relpath"] not in done]
print(f"이미 끝난 세그먼트 {len(done)} · 남은 {len(todo)}")


def transcribe(clips):
    feats = processor.feature_extractor(clips, sampling_rate=W.SR, return_tensors="pt").input_features
    with torch.no_grad():
        ids = model.generate(input_features=feats.cuda())
    return processor.tokenizer.batch_decode(ids, skip_special_tokens=True)


t0, n_kept, n_cand = time.time(), 0, 0
with open(OUT, "a", encoding="utf-8") as f:
    for i, r in enumerate(todo, 1):
        audio, sr = sf.read(os.path.join(REF, r["segment_wav_relpath"]), dtype="float32")
        if audio.ndim > 1:
            audio = audio.mean(1)
        assert sr == W.SR, sr
        try:
            cands = W.cut_one(aligner, audio, r, WORDS)
        except Exception as e:                  # 한 세그먼트 때문에 전체가 멈추지 않게 — 이유를 남긴다
            cands = [{"segment": r["segment_wav_relpath"], "parent": r["parent_file_id"],
                      "word": None, "occ": -1, "sentence": r["reference_text"],
                      "skip": f"error: {type(e).__name__}: {e}"[:200]}]
        live = [c for c in cands if "skip" not in c]
        if live:
            hyps = transcribe([audio[int(c["t0"] * W.SR): int(c["t1"] * W.SR)] for c in live])
            for c, h in zip(live, hyps):
                c["hyp"] = h.strip()
                c["keep"] = W.keep(c["word"], h)
        for c in cands:
            f.write(json.dumps(c, ensure_ascii=False) + "\n")
        f.flush()
        n_cand += len(live)
        n_kept += sum(c["keep"] for c in live)
        if i % 200 == 0 or i == len(todo):
            el = time.time() - t0
            print(f"  {i}/{len(todo)} · 후보 {n_cand} · 남김 {n_kept} ({n_kept / max(n_cand, 1):.0%})"
                  f" · {el / 60:.1f}분 · 남은 약 {el / i * (len(todo) - i) / 60:.0f}분", flush=True)
print("완료" if len(done) + len(todo) == len(plan) else "")

## 셀 6 — 요약 (이 출력을 붙여 주세요)

In [ ]:
from collections import Counter, defaultdict

res = [json.loads(l) for l in open(OUT, encoding="utf-8") if l.strip()]
live = [c for c in res if "skip" not in c]
kept = [c for c in live if c["keep"]]
print(f"세그먼트 {len({c['segment'] for c in res})}/{len(plan)} · 단어 자리 {len(res)} · 후보 {len(live)} · 남김 {len(kept)} ({len(kept) / max(len(live), 1):.0%})")
print("건너뜀:", dict(Counter(c["skip"].split(":")[0] for c in res if "skip" in c)))
print(f"토큰 경계 clean: 후보 {sum(c['clean'] for c in live)} · 남김 {sum(c['clean'] for c in kept)}")
for n in (1, 2, 3):
    lv = [c for c in live if len(c["word"]) == n] if n < 3 else [c for c in live if len(c["word"]) >= 3]
    kp = [c for c in lv if c["keep"]]
    print(f"  {n}{'음절' if n < 3 else '음절 이상'}: 후보 {len(lv)} · 남김 {len(kp)} ({len(kp) / max(len(lv), 1):.0%})")

by = defaultdict(lambda: [0, 0])
for c in live:
    by[c["word"]][0] += 1
    by[c["word"]][1] += c["keep"]
have = sorted(((k, v) for k, v in by.items() if v[1] > 0), key=lambda kv: -kv[1][1])
print(f"\n남은 클립이 있는 앱 단어 {len(have)}종 / 후보가 있던 단어 {len(by)}종")
print("  많은 순:", ", ".join(f"{w} {k}/{n}" for w, (n, k) in have[:30]))
print("  0개:", ", ".join(f"{w}({n})" for w, (n, k) in sorted(by.items(), key=lambda kv: -kv[1][0]) if k == 0)[:600])
from cut_word_segments import PART
dropped = [c for c in live if not c["keep"]]
with_particle = [c for c in dropped
                 if W.norm(c["hyp"]).startswith(c["word"]) and W.norm(c["hyp"])[len(c["word"]):] in PART]
print()
print(f"버린 {len(dropped)}개 중 '단어+조사'로 들린 것 {len(with_particle)}개 ({len(with_particle) / max(len(dropped), 1):.0%})"
      " — 경계가 조사까지 먹은 손실. 크면 조사 쪽 경계를 다듬을 여지가 있다")
print(f"  조사가 붙는 자리: 후보 {sum(c['particle'] for c in live)} · 남김 {sum(c['particle'] for c in kept)}"
      f" / 어절 끝: 후보 {sum(not c['particle'] for c in live)} · 남김 {sum(not c['particle'] for c in kept)}")
wrong = Counter((c["word"], c["hyp"]) for c in live if not c["keep"])
print("\n버린 것 중 잦은 받아쓰기:", ", ".join(f"{w}→{h!r} {n}" for (w, h), n in wrong.most_common(20)))
summary = {"segments_done": len({c['segment'] for c in res}), "plan": len(plan), "candidates": len(live),
           "kept": len(kept), "words_with_clips": len(have), "words_with_candidates": len(by),
           "dropped_word_plus_particle": len(with_particle)}
json.dump(summary, open(f"{D}/ts01rest_word_cuts_summary.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print("\n→ 로컬로 받을 것: MyDrive/ts01rest_word_cuts.jsonl (+ 아직 안 받았다면 ts01rest_segs_refined.zip)")